# 02 - Run the TabSyn baseline (Colab)

Runs `eval/run_tabsyn_baseline.py` over all four variants to produce the comparison
numbers in Table II of the letter.

**Why this is chunked.** TabSyn is expensive: on the Comprehensive Joint Set it costs
888.52 s per iteration to train, against 12.60 s for our model. Summed over the four
variants, one iteration is roughly 4,794 s, so the 20 iterations used in the letter
come to about **27 hours of training** - far longer than a single Colab session. Set
`START`/`END` below and run the notebook once per chunk. `--end` is exclusive.

The four chunks used for the letter, which reproduce the committed filenames exactly:

| START | END | produces |
|---|---|---|
| 0 | 2 | `tabsyn_baseline_0_to_1.csv` |
| 2 | 4 | `tabsyn_baseline_2_to_3.csv` |
| 4 | 10 | `tabsyn_baseline_4_to_9.csv` |
| 10 | 20 | `tabsyn_baseline_10_to_19.csv` |

**Iteration `i` uses `seed = 42 + i`, identical to the Flow Matching pipeline**, so
run `i` of each method sees the same train/val/test split. That is what makes the
AUCs directly comparable.

**Prerequisites** the same Drive layout as notebook `01`, with `DRIVE_ROOT` pointing at
your own folder, plus a GPU runtime.

**Produces** `results/tabsyn/<variant>/tabsyn_baseline_<START>_to_<END-1>.csv`.

In [ ]:
# ==========================================
# Configuration
# ==========================================
START = 0                       # first iteration (0-indexed)
END = 2                         # exclusive
REPO_URL = "https://github.com/abstratovcm/pruned-soil-fm.git"
REPO_DIR_BASE = "pruned-soil-fm"
DRIVE_ROOT = "/content/drive/MyDrive/pruned-soil-fm"   # your own folder
# ==========================================
SUFFIX = f"{START:02d}_to_{END:02d}"
REPO_DIR = f"{REPO_DIR_BASE}_{SUFFIX}"
RUN_NAME = f"tabsyn_{SUFFIX}"   # an existing folder is overwritten
print(f"Configured for iterations {START}..{END - 1} -> Drive folder: {RUN_NAME}")
print(f"Will write: tabsyn_baseline_{START}_to_{END - 1}.csv")

In [ ]:
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# The only dependencies Colab does not already ship.
!pip install -q -r "/content/{REPO_DIR}/requirements-tabsyn.txt"
# zero pins Click==7.0; ply and quantiphy are what its import needs.
!pip install -q --no-deps zero ply quantiphy

In [ ]:
# Link the checkout to Drive, and clone the TabSyn baseline into it.
repo_path = f"/content/{REPO_DIR}"
run_path = f"{DRIVE_ROOT}/{RUN_NAME}"

# 1. Prepared datasets (shared, read-only)
!mkdir -p "{repo_path}/data"
!rm -rf "{repo_path}/data/prepared_datasets"
!ln -s "{DRIVE_ROOT}/prepared_datasets" "{repo_path}/data/prepared_datasets"

# 2. TabSyn source, cloned inside this checkout so parallel chunks stay isolated
%cd "{repo_path}"
!git clone -q https://github.com/amazon-science/tabsyn.git

# 3. Baseline AUC outputs -> Drive
!mkdir -p "{run_path}/results_tabsyn"
!mkdir -p "{repo_path}/results"
!rm -rf "{repo_path}/results/tabsyn"
!ln -s "{run_path}/results_tabsyn" "{repo_path}/results/tabsyn"

# 4. TabSyn's working directory -> Drive; holds the generated synthetic_*.csv
!mkdir -p "{run_path}/tabsyn_workdir"
!rm -rf "{repo_path}/tabsyn/data"
!ln -s "{run_path}/tabsyn_workdir" "{repo_path}/tabsyn/data"

!ls -la "{repo_path}/results"

In [ ]:
# Required: TabSyn passes verbose=True to a scheduler signature that no
# longer accepts it, and its VAE training raises TypeError unpatched.
!bash "{repo_path}/scripts/patch_tabsyn.sh"

In [ ]:
%cd {repo_path}
!PYTHONPATH=. python eval/run_tabsyn_baseline.py --start {START} --end {END}